In [64]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


##1.Load the Dataset
Load the dataset named "trump_tweet_sentiment_analysis.csv" using pandas. Ensure the dataset contains at least two columns: "text" and "label".

In [66]:
df = pd.read_csv("/content/drive/MyDrive/AI ML/Workshop8/trum_tweet_sentiment_analysis.csv")

In [67]:
df.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [68]:
df.isnull().sum()

,0
text,0
Sentiment,0


##2.Text Cleaning and Tokenization
Apply a text preprocessing pipeline to the "text" column. This should include:

Lowercasing the text
Removing URLs, mentions, punctuation, and special characters
Removing stopwords
Tokenization (optional: stemming or lemmatization)
"Complete the above function"

In [69]:
def lower_order(text):
  """
  This function converts all the text in input text to lower order.
  Input Args:
  token_text : input text.
  Returns:
  small_order_text : text converted to small/lower order.
  """
  small_order_text = text.lower()
  return small_order_text

In [70]:
import re

#Removing urls from text
def remove_urls(text):
  return re.sub(r'http\S+', '', text)

In [91]:
# Remove mentions (e.g., @username)
def remove_mentions(text):
  data = re.sub(r'@\w+', '', text)
  return data

In [72]:
#Remove emojis
def remove_emojis(text):
  new = ""
  for i in text:
    if i.isascii():
      new += i
  return new

In [73]:
import string

#Remove punctuations form text
def remove_punctuations(text):
  return text.translate(str.maketrans('', '', string.punctuation))

In [74]:
#Remove numbers
def remove_numbers(text):
  new = ""
  for i in text:
    if not i.isdigit():
      new += i
  return new

In [85]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

In [86]:
stop_words = set(stopwords.words('english'))

In [87]:
#Tokenize means splitting words from sentence in array form
#checking the words are stopwords or not
def remove(text):
  # Create tokens.
  words = word_tokenize(text)
  cleaned = []
  for i in words:
    if i not in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)

In [92]:

def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This function cleans text data by lowercasing, removing URLs, mentions, emojis, and
  other unwanted characters, then tokenizes and removes stopwords, and finally
  applies lemmatization or stemming.

  Args:
    dataset (str): The input text string to be cleaned.
    rule (str, optional): The rule for word reduction. Can be "lemmatize" or "stem".
                          Defaults to "lemmatize".

  Returns:
    str: The cleaned and processed text.
  """
  # Convert the input to small/lower order.
  data = lower_order(dataset)

  # Remove URLs
  data = remove_urls(data)

  # Remove mentions (e.g., @username)
  data = remove_mentions(data)

  # Remove emojis
  data = remove_emojis(data)

  # Remove all other unwanted characters (punctuation and numbers)
  data = remove_punctuations(data)
  data = remove_numbers(data)

  # Remove stopwords:
  data = remove(data)

  # Tokenize the text
  tokens = nltk.word_tokenize(data)

  if rule == "lemmatize":
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")
    return ""

  return " ".join(tokens)

In [93]:
cleaned_data = df["text"].apply(text_cleaning_pipeline)
cleaned_data.head()

,text
0,rt trump draining swamp taxpayer dollar trip a...
1,icymi hacker rig fm radio station play antitru...
2,trump protest lgbtq rally new york bbcworld via
3,hi im pier morgan david beckham awful donald t...
4,rt tech firm suing buzzfeed publishing unverif...


##3.Train-Test Split

In [97]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(cleaned_data, df['Sentiment'], test_size=0.2, random_state=42)

##4.TF-IDF Vectorization

In [98]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)

##5.Model Training and Evaluation

In [105]:
#Using linear regression

from sklearn.linear_model import LogisticRegression


lg_model = LogisticRegression(max_iter=1000)

lg_model.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [106]:
lg_pred = lg_model.predict(x_test_tfidf)
print(accuracy_score(y_test, lg_pred))

0.9515573272076211
